# Testování hypotéz

Tento notebook testuje dvě hypotézy pomocí dat o nehodách a vozidlech:

1. **Hypotéza 1**: Na silnicích první třídy se byly nehody s následky na zdraví se stejnou pravděpodobností jako na dálnicích.
   - Metoda: Chi-kvadrát test (𝜒²).
2. **Hypotéza 2**: Škoda při nehodách trolejbusů je nižší než při nehodách autobusů a tento rozdíl je statisticky významný.
   - Metoda: Vhodný test vybraný na základě distribuce dat.

K analýze využívám následující datové soubory:
- `accidents.pkl.gz`
- `vehicles.pkl.gz`


In [6]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# load data
df_accidents = pd.read_pickle("accidents.pkl.gz")

# filter data
highway_accidents = df_accidents[df_accidents['p36'] == 0]  # highways
primary_road_accidents = df_accidents[df_accidents['p36'] == 1]  # primary roads

# count the number of accidents with and without injuries for both road types
highway_injuries = highway_accidents['p9'].value_counts()
primary_road_injuries = primary_road_accidents['p9'].value_counts()

# cceate a contingency table for Chi-squared test
contingency_table = pd.DataFrame({
    "Dálnice": highway_injuries,
    "Silnice první třídy": primary_road_injuries
}).fillna(0)
print("Kontingenční tabulka:")
print(contingency_table)

# perform the Chi-squared test
chi2, p, dof, expected = chi2_contingency(contingency_table)

# results of test
print(f"\nChi-kvadrát: {chi2}, p-hodnota: {p}")
print("Očekávané hodnoty:")
print(expected)

# interpret results
if p < 0.05:
    print("Nulová hypotéza je zamítnuta")
else:
    print("Nepodařilo se zamítnout nulovou hypotézu")




Kontingenční tabulka:
    Dálnice  Silnice první třídy
p9                              
2      6674                14773
1      1247                 7059

Chi-kvadrát: 794.1532980454858, p-hodnota: 1.0075002730159729e-174
Očekávané hodnoty:
[[ 5709.73303532 15737.26696468]
 [ 2211.26696468  6094.73303532]]
Nulová hypotéza je zamítnuta


## Výsledky Hypotézy 1

Kontingenční tabulka:
p9 (následky na zdraví)  Dálnice  Silnice 1. třídy 
2 (bez zranění)           6674     14773           
1 (s následky na zdraví)  1247     7059            

Chi-kvadrát test ukázal následující hodnoty:
- **Chi-kvadrát**: 794.15
- **p-hodnota**: 1.007e-174
- **Očekávané hodnoty**:
    - Pro nehody bez zranění: 5709.73 (dálnice) a 15737.27 (silnice 1. třídy)
    - Pro nehody s následky: 2211.27 (dálnice) a 6094.73 (silnice 1. třídy)

### Závěr
- **Nulová hypotéza je zamítnuta**. 
- Nehody s následky na zdraví nejsou stejně pravděpodobné na dálnicích a silnicích 1. třídy.
- Silnice 1. třídy vykazují vyšší počet nehod s následky na zdraví.


In [4]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, normaltest

# load data
df_vehicles = pd.read_pickle("vehicles.pkl.gz")

# filter data for trolleybuses and buses
trolleybus_damage = df_vehicles[df_vehicles['p44'] == 10]['p53']
bus_damage = df_vehicles[df_vehicles['p44'] == 8]['p53']

# check for normality in both groups
trolleybus_normality = normaltest(trolleybus_damage)
bus_normality = normaltest(bus_damage)

print("Normálnost dat:")
print(f"Trolejbusy: stat={trolleybus_normality.statistic}, p={trolleybus_normality.pvalue}")
print(f"Autobusy: stat={bus_normality.statistic}, p={bus_normality.pvalue}")

# if data is not normally distributed use the Mann-Whitney U test
if trolleybus_normality.pvalue < 0.05 or bus_normality.pvalue < 0.05:
    test_statistic, p_value = mannwhitneyu(trolleybus_damage, bus_damage, alternative="two-sided")
    print("\nPoužitý test: Mann-Whitney U test")
    print(f"Statistika: {test_statistic}, p-hodnota: {p_value}")
    
    # interpret results
    if p_value < 0.05:
        if trolleybus_damage.median() > bus_damage.median():
            print("Škoda při nehodách trolejbusů je vyšší než u autobusů.")
        else:
            print("Škoda při nehodách trolejbusů je nižší než u autobusů.")
    else:
        print("Rozdíl ve škodě mezi trolejbusy a autobusy není statisticky významný.")
else:
    print("Data jsou normálně rozložena. Měli bychom použít parametrický test (např. t-test).")

# calculate medians for summary
print("\nMediány škody:")
print(f"Trolejbusy: {trolleybus_damage.median()} Kč")
print(f"Autobusy: {bus_damage.median()} Kč")




Normálnost dat:
Trolejbusy: stat=1929.4727066249993, p=0.0
Autobusy: stat=7458.043692568776, p=0.0

Použitý test: Mann-Whitney U test
Statistika: 2683349.5, p-hodnota: 6.431370775779769e-22
Škoda při nehodách trolejbusů je vyšší než u autobusů.

Mediány škody:
Trolejbusy: 400.0 Kč
Autobusy: 200.0 Kč


## Výsledky Hypotézy 2

### Test normálnosti dat
- **Trolejbusy**: stat = 1929.47, p-hodnota = 0.0
- **Autobusy**: stat = 7458.04, p-hodnota = 0.0

Data nejsou normálně rozložena, použil se **Mann-Whitney U test**.

### Výsledky Mann-Whitney U testu
- Statistika: 2683349.5
- p-hodnota: 6.43e-22
- Závěr: Škoda při nehodách trolejbusů je statisticky významně **vyšší** než u autobusů.

### Mediány škody
- **Trolejbusy**: 400 Kč
- **Autobusy**: 200 Kč

### Závěr
Nulová hypotéza je zamítnuta. Škoda při nehodách trolejbusů je **významně vyšší** než škoda při nehodách autobusů.
